<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.it/cap05/cap05.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 💻 **Parte Pratica con Esercizi di Programmazione**

La presente lista di esercizi di programmazione (EP) consolida le formulazioni teoriche presentate nel corso del Capitolo 5 — Trasformate e Compressione — attraverso un percorso pratico applicato. Gli esercizi sono strutturati a partire da matrici di dimensioni ridotte, consentendo la validazione analitica e l'ispezione manuale di ciascun coefficiente, mantenendo la coerenza metodologica adottata nei capitoli precedenti.

L'incatenamento degli esercizi riproduce rigorosamente il flusso concettuale del capitolo: si inizia con l'implementazione esplicita della Trasformata Discreta di Fourier (DFT) a partire dalla sua definizione matematica fondamentale; si prosegue con la progettazione di filtri passa-basso e maschere *notch* nel dominio della frequenza; si applica la quantizzazione dei coefficienti (nucleo della compressione con perdita); e si conclude con l'integrazione di queste fasi nella costruzione di un *pipeline* di compressione JPEG semplificato e nell'analisi percettiva dei formati immagine.

> ### ❗ Linee Guida per la Risoluzione degli Esercizi di Programmazione
>
> In tutti gli esercizi di questo capitolo, le coordinate del **centro dello spettro** (origine delle frequenze spaziali dopo l'applicazione dello spostamento `fftshift`) devono essere determinate tramite divisione intera. Per una matrice con $L$ righe e $C$ colonne, la componente di frequenza nulla si trova nella posizione:
>
> $$
> (c_y, c_x) = \left( \left\lfloor \frac{L}{2} \right\rfloor, \left\lfloor \frac{C}{2} \right\rfloor \right)
> $$
>
> Questa convenzione è rigorosamente identica a quella adottata dalla funzione `np.fft.fftshift`. Inoltre, in tutte le fasi che richiedono discretizzazione o arrotondamento numerico (sia nella quantizzazione dei coefficienti AC sia nella ricostruzione finale dei pixel), si deve impiegare l'arrotondamento standard al numero intero più vicino (*round half away from zero*), mitigando ambiguità in valori con frazione esattamente pari a $0.5$.

### 🎯 Obiettivo di questo Quaderno

Il quaderno consente di sviluppare, validare, organizzare e testare soluzioni di **Esercizi di Programmazione (EPs)** in ambienti interattivi, come Colab, con gli stessi casi di test di Moodle, copiandoli lì solo al momento di registrare il voto ufficiale.

#### *Download*

Scarica `morph.py` e `testsuite.py` eseguendo la cella qui sotto:

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Esecuzione dei Test
Per valutare i test, esegui `TestSuite("EP05_01.estensione").run()` in una nuova cella, sostituendo l'estensione con quella del linguaggio utilizzato (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). Il sistema scarica i casi di test da GitHub, esegue il programma e calcola automaticamente il voto.

Per testare il codice Python direttamente, senza salvare un file, usa `run_code(codice)` passando il codice come *stringa* in una variabile `codice`:

```python
codice = """
from morph import mm
# ... il tuo codice qui ...
"""
TestSuite("EP05_01").run_code(codice)
```

### EP05_01 🟢 Filtro Passa-Basso Ideale per Distanza nello Spettro

In uno ***scanner* di documenti antico**, il sensore cattura carta stropicciata e la trama delle fibre insieme al testo — rumore ad alta frequenza che "inquina" lo spettro ai bordi. Il tecnico della manutenzione non ha accesso all'immagine originale, ma solo allo **spettro di magnitudo già calcolato** dal software dello *scanner*. Il suo compito è semplice e chirurgico: mantenere solo il **cerchio centrale** delle basse frequenze (la struttura globale del documento) ed eliminare tutto ciò che si trova al di fuori del raggio $D_0$, rimuovendo la trama fine senza nemmeno dover toccare l'immagine spaziale.

Questo è il **Filtro Passa-Basso Ideale (LPFI)**: l'operazione spettrale più diretta del capitolo, ma anche quella che meglio rivela l'anatomia di uno spettro centrato.

#### 📋 Linee Guida di Implementazione

1. **Dimensioni:** Leggere gli interi $L$ (righe) e $C$ (colonne) dello spettro di magnitudo — già fornito **centrato** (equivalente all'uscita di `np.fft.fftshift`).
2. **Frequenza di taglio:** Leggere l'intero $D_0$.
3. **Dati:** Leggere i valori interi della matrice di magnitudo, riga per riga.
4. **Centro dello spettro:** Calcolare $(c_y, c_x) = (L \mathbin{//} 2,\; C \mathbin{//} 2)$.
5. **Distanza:** Per ogni posizione $(u,v)$, calcolare
$$
D(u,v) = \sqrt{(u-c_y)^2 + (v-c_x)^2}
$$
6. **Maschera ideale:** Applicare
$$
H(u,v) = \begin{cases} 1, & D(u,v) \le D_0 \\ 0, & D(u,v) > D_0 \end{cases}
$$
7. **Filtraggio:** Il valore di uscita è $\text{mag}'(u,v) = \text{mag}(u,v) \cdot H(u,v)$.
8. **Uscita:** Visualizzare la matrice filtrata con dimensioni $L \times C$.

#### 📌 Vincoli Computazionali

* **Confronto non stretto:** il criterio usa $D(u,v) \le D_0$ (il confine appartiene al filtro, cioè viene mantenuto).
* **Tipo:** tutti i valori di ingresso e uscita sono interi; la distanza è calcolata in virgola mobile solo internamente.
* **Nessun arrotondamento della magnitudo:** poiché l'ingresso è già intero e la maschera è binaria (0 o 1), l'uscita non richiede mai arrotondamento.

#### 🧠 Fondamenti Teorici

| Regione | Distanza dal centro | Effetto del filtro |
|---|---|---|
| **Centro** ($D \le D_0$) | Basse frequenze | Preservate — struttura globale mantenuta |
| **Bordi** ($D > D_0$) | Alte frequenze | Azzerate — trama e rumore rimossi |
| **$D_0$ piccolo** | — | L'immagine ricostruita sarebbe molto sfocata |
| **$D_0$ grande** | — | Poca filtrazione; quasi tutta l'energia preservata |

#### 📦 Specifica di Ingresso e Uscita (VPL)

**Ingresso:**

* Riga 1: Intero $L$.
* Riga 2: Intero $C$.
* Riga 3: Intero $D_0$.
* Righe successive: Elementi interi della matrice di magnitudo (centrata).

**Uscita:**

* Matrice filtrata con $L$ righe e $C$ colonne, separati da spazi.

#### 📌 Esempi

| Ingresso | Uscita | Osservazione |
|---|---|---|
| 3<br>3<br>1<br>10 20 30<br>40 50 60<br>70 80 90 | 0 20 0<br>40 50 60<br>0 80 0 | Centro $(1,1)$. Gli angoli hanno $D=\sqrt{2}\approx1.41 > 1$, quindi vengono azzerati; i vicini ortogonali hanno $D=1 \le 1$ e sono mantenuti. |
| 1<br>3<br>0<br>5 9 7 | 0 9 0 | $L=1, C=3$: centro in $(0,1)$. Solo la posizione centrale stessa ($D=0$) sopravvive a $D_0=0$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0501" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0501 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0501 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0501 button:hover { background: #e8dfcf; }
  #sim-ep0501 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0501_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0501_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0501_grid_stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-bottom: 12px; }
  .sim-ep0501_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .sim-ep0501_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim-ep0501_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim-ep0501_cell { width: 42px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP05_01: Filtro Passa-Basso Ideale</span>
  <span class="sim-ep0501_pill">H = (D &le; D₀) ? 1 : 0</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0501_panel" style="margin-bottom:14px;">
    
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Raggio di taglio (D₀): <span id="sim-ep0501_vl_d0" style="font-family:monospace; color:#26241d;">1</span>
      </label>
    </div>
    
    <input id="sim-ep0501_sl_d0" type="range" min="0" max="4" step="1" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Regola D₀ e osserva quali posizioni dello spettro 5&times;5 sopravvivono al filtro.
    </div>

  </div>

  <!-- Exibição das Grades de Espectro -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0501_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Spettro Originale (Magnitudine)
      </div>
      <div id="sim-ep0501_grid_orig" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0501_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Risultato Filtrato
      </div>
      <div id="sim-ep0501_grid_new" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0501_debug" class="sim-ep0501_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    –
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep01(root){
    if (!root || root.dataset.sim05Ep01Init) return;
    root.dataset.sim05Ep01Init = "1";

    var N = 5, cy = Math.floor(N / 2), cx = Math.floor(N / 2);
    var mag = [];
    for (var i = 0; i < N; i++){
      var row = [];
      for (var j = 0; j < N; j++){
        row.push(10 * (i + 1) + j + 1);
      }
      mag.push(row);
    }

    var d0el = root.querySelector('#sim-ep0501_sl_d0');
    var d0v  = root.querySelector('#sim-ep0501_vl_d0');
    var go   = root.querySelector('#sim-ep0501_grid_orig');
    var gn   = root.querySelector('#sim-ep0501_grid_new');
    var dbg  = root.querySelector('#sim-ep0501_debug');

    function cellStyle(active){
      if (active) {
        return 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      } else {
        return 'background:#fafaf7; color:#8a8371; border:1px solid #e4dcc8;';
      }
    }

    function render(){
      var D0 = parseInt(d0el.value, 10);
      d0v.textContent = D0;
      go.innerHTML = '';
      gn.innerHTML = '';
      var kept = 0;

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var d = Math.sqrt((i - cy) * (i - cy) + (j - cx) * (j - cx));
          var keep = d <= D0;
          if (keep) kept++;

          var co = document.createElement('div');
          co.className = 'sim-ep0501_cell';
          co.style.cssText = cellStyle(true);
          co.textContent = mag[i][j];
          go.appendChild(co);

          var cn = document.createElement('div');
          cn.className = 'sim-ep0501_cell';
          cn.style.cssText = cellStyle(keep);
          cn.textContent = keep ? mag[i][j] : 0;
          gn.appendChild(cn);
        }
      }

      dbg.textContent = 'Centro = (' + cy + ', ' + cx + ')  |  D₀ = ' + D0 + '  |  Coeficientes mantidos: ' + kept + ' / ' + (N * N);
    }

    d0el.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep01(){
    var root = document.getElementById('sim-ep0501');
    if (root) initSim05Ep01(root); else setTimeout(tryInitSim05Ep01, 200);
  }
  tryInitSim05Ep01();
})();
</script>
""")

**Figura 5.1:** Simulatore EP05_01: Filtro Passa-Basso Ideale nello Spettro


<figure id="fig-05-sim-ep0501">
  <img src="imagens/fig-05-sim-ep0501.png" alt=" Simulatore EP05_01: Filtro Passa-Basso Ideale nello Spettro " style="max-width:80%" />
  <figcaption><strong>Figura 5.1:</strong>  Simulatore EP05_01: Filtro Passa-Basso Ideale nello Spettro </figcaption>
</figure>

In [ ]:
%%writefile EP05_01.py
# Codice Python

In [ ]:
TestSuite("EP05_01.py").run()

### EP05_02 🟡 Filtro *Notch*: Rimozione dei Picchi Periodici

Una telecamera di **ispezione industriale** acquisisce immagini di circuiti stampati, ma l'alimentazione della linea di produzione introduce un'**interferenza elettrica periodica** — un pattern di strisce quasi impercettibile a occhio nudo, che però appare nello spettro di Fourier come **coppie di picchi luminosi** posizionati simmetricamente attorno al centro. Il team di visione artificiale non può rielaborare l'acquisizione: deve **localizzare e cancellare chirurgicamente** queste coppie di picchi nello spettro, preservando tutta l'altra informazione utile dell'immagine.

Questo è il ruolo del **filtro rigetta-banda *notch***: a differenza del passa-basso (che interessa una regione continua), esso agisce su **punti specifici e sui loro simmetrici**, lasciando intatto il resto dello spettro.

#### 📋 Linee Guida di Implementazione

1. **Dimensioni:** Leggere gli interi $L$ (righe) e $C$ (colonne) dello spettro di magnitudine centrato.
2. **Dati:** Leggere i valori interi della matrice di magnitudine, riga per riga.
3. **Picchi:** Leggere l'intero $K$ (numero di coppie di picchi da rimuovere).
4. **Per ciascuno dei $K$ picchi:** leggere tre interi $\Delta v$, $\Delta u$, $r$ — spostamento verticale, spostamento orizzontale e raggio del *notch*.
5. **Centro dello spettro:** $(c_y, c_x) = (L \mathbin{//} 2,\; C \mathbin{//} 2)$.
6. **Soppressione simmetrica:** per ogni picco, azzerare **tutte** le posizioni $(u,v)$ tali che la distanza dal punto $(c_y+\Delta v,\, c_x+\Delta u)$ sia $\le r$, **e anche** tutte le posizioni con distanza $\le r$ dal punto simmetrico $(c_y-\Delta v,\, c_x-\Delta u)$.
7. **Uscita:** Visualizzare la matrice risultante con dimensioni $L \times C$.

#### 📌 Vincoli Computazionali

* **Simmetria obbligatoria:** ogni picco indicato genera **due** dischi azzerati (il punto e il suo simmetrico rispetto al centro) — dimenticare il simmetrico è l'errore più comune.
* **Sovrapposizione:** se due dischi si sovrappongono, la posizione rimane azzerata (non c'è "somma" o ripristino).
* **Confronto non stretto:** una posizione viene azzerata se $\text{distanza} \le r$.
* **Ordine di lettura:** i $K$ picchi devono essere elaborati nell'ordine in cui compaiono nell'input, ma il risultato finale non dipende dall'ordine (le operazioni di azzeramento sono commutative).

#### 🧠 Fondamenti Teorici

| Concetto | Ruolo nel filtro *notch* |
|---|---|
| **Picco in $(\Delta v, \Delta u)$** | Frequenza dell'interferenza periodica rilevata visivamente nello spettro |
| **Punto simmetrico $(-\Delta v,-\Delta u)$** | Ogni DFT di segnale reale è hermitiana: i picchi compaiono sempre in coppie simmetriche rispetto al centro |
| **Raggio $r$** | Controlla la "larghezza" della reiezione — un $r$ grande rimuove più energia attorno al picco, ma anche informazione utile |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $L$.
* Riga 2: Intero $C$.
* Righe successive: Elementi interi della matrice di magnitudine (centrata), $L$ righe.
* Riga successiva: Intero $K$.
* $K$ righe successive: tre interi $\Delta v$, $\Delta u$, $r$ (separati da spazi).

**Output:**

* Matrice risultante in $L$ righe e $C$ colonne, separati da spazi.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 5<br>5<br>1 2 3 4 5<br>6 7 8 9 10<br>11 12 13 14 15<br>16 17 18 19 20<br>21 22 23 24 25<br>1<br>1 1 0 | 1 2 3 4 5<br>6 0 8 9 10<br>11 12 13 14 15<br>16 17 18 0 20<br>21 22 23 24 25 | Centro $(c_y, c_x) = (2, 2)$. Il picco indicato $(\Delta v, \Delta u) = (1, 1)$ genera il punto $(3, 3)$ (valore 19) e il suo simmetrico $(1, 1)$ (valore 7), entrambi azzerati con $r=0$ (solo i punti esatti). |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0502" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0502 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0502 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0502 button:hover { background: #e8dfcf; }
  #sim-ep0502 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0502_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0502_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0502_grid_ctrls { display: grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr)); gap: 12px; }
  .sim-ep0502_cell { width: 42px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP05_02: Filtro Notch</span>
  <span class="sim-ep0502_pill">Coppia simmetrica</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0502_panel" style="margin-bottom:14px;">
    <div class="sim-ep0502_grid_ctrls">
      
      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">&Delta;v</label>
          <span id="sim-ep0502_vl_dv" style="font-family:monospace; font-weight:700; color:#26241d;">1</span>
        </div>
        <input id="sim-ep0502_sl_dv" type="range" min="-2" max="2" step="1" value="1">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">&Delta;u</label>
          <span id="sim-ep0502_vl_du" style="font-family:monospace; font-weight:700; color:#26241d;">1</span>
        </div>
        <input id="sim-ep0502_sl_du" type="range" min="-2" max="2" step="1" value="1">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Raggio (r)</label>
          <span id="sim-ep0502_vl_r" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim-ep0502_sl_r" type="range" min="0" max="2" step="1" value="0">
      </div>

    </div>

    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:10px; text-align:center;">
      Muovi &Delta;v e &Delta;u per scegliere il picco &mdash; osserva che anche la coppia simmetrica viene filtrata.
    </div>
  </div>

  <!-- Espectro 5x5 -->
  <div class="sim-ep0502_panel" style="text-align:center; margin-bottom:14px;">
    <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
      Spettro 5&times;5 (Rosso = Rimosso dal Filtro)
    </div>
    <div id="sim-ep0502_grid" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0502_debug" class="sim-ep0502_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep02(root){
    if (!root || root.dataset.sim05Ep02Init) return;
    root.dataset.sim05Ep02Init = "1";

    var N = 5, cy = Math.floor(N / 2), cx = Math.floor(N / 2);
    var mag = [];
    for (var i = 0; i < N; i++){
      var row = [];
      for (var j = 0; j < N; j++){
        row.push(i * 5 + j + 1);
      }
      mag.push(row);
    }

    var dv  = root.querySelector('#sim-ep0502_sl_dv');
    var du  = root.querySelector('#sim-ep0502_sl_du');
    var r   = root.querySelector('#sim-ep0502_sl_r');
    var dvv = root.querySelector('#sim-ep0502_vl_dv');
    var duv = root.querySelector('#sim-ep0502_vl_du');
    var rv  = root.querySelector('#sim-ep0502_vl_r');

    var grid = root.querySelector('#sim-ep0502_grid');
    var dbg  = root.querySelector('#sim-ep0502_debug');

    function cellStyle(kill){
      if (kill) {
        return 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
      } else {
        return 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      }
    }

    function render(){
      var DV = parseInt(dv.value, 10);
      var DU = parseInt(du.value, 10);
      var R  = parseInt(r.value, 10);

      dvv.textContent = DV;
      duv.textContent = DU;
      rv.textContent  = R;

      var p1 = [cy + DV, cx + DU];
      var p2 = [cy - DV, cx - DU];

      grid.innerHTML = '';
      var removed = 0;

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var d1 = Math.sqrt((i - p1[0]) * (i - p1[0]) + (j - p1[1]) * (j - p1[1]));
          var d2 = Math.sqrt((i - p2[0]) * (i - p2[0]) + (j - p2[1]) * (j - p2[1]));
          var kill = (d1 <= R) || (d2 <= R);

          if (kill) removed++;

          var c = document.createElement('div');
          c.className = 'sim-ep0502_cell';
          c.style.cssText = cellStyle(kill);
          c.textContent = kill ? 0 : mag[i][j];
          grid.appendChild(c);
        }
      }

      dbg.textContent = 'Centro = (' + cy + ', ' + cx + ')  |  Pico = (' + p1[0] + ', ' + p1[1] + ')  |  Simétrico = (' + p2[0] + ', ' + p2[1] + ')  |  Removidos: ' + removed;
    }

    [dv, du, r].forEach(function(el){
      el.addEventListener('input', render);
    });

    render();
  }

  function tryInitSim05Ep02(){
    var root = document.getElementById('sim-ep0502');
    if (root) initSim05Ep02(root); else setTimeout(tryInitSim05Ep02, 200);
  }
  tryInitSim05Ep02();
})();
</script>
""")

**Figura 5.2:** Simulatore EP05_02: Filtro Notch


<figure id="fig-05-sim-ep0502">
  <img src="imagens/fig-05-sim-ep0502.png" alt=" Simulatore EP05_02: Filtro Notch " style="max-width:80%" />
  <figcaption><strong>Figura 5.2:</strong>  Simulatore EP05_02: Filtro Notch </figcaption>
</figure>

In [ ]:
%%writefile EP05_02.py
# Codice Python

In [ ]:
TestSuite("EP05_02.py").run()

### EP05_03 🟠 Quantizzazione DCT: la Vera Fonte di Compressione

Un'applicazione di **galleria fotografica** deve ridurre le dimensioni di migliaia di immagini prima di caricarle sul cloud, senza ricodificare tutto da zero. L'ingegnere responsabile ha già i **coefficienti DCT** di ogni blocco $4\times4$ calcolati (la fase computazionalmente onerosa è già stata eseguita) — manca solo applicare la **tabella di quantizzazione**, la fase che scarta realmente informazioni e genera compressione. I coefficienti ad alta frequenza, meno percettibili all'occhio umano, ricevono divisori grandi e tendono a diventare **zero**; i coefficienti a bassa frequenza, più percettibili, ricevono divisori piccoli e sopravvivono quasi intatti.

Dovrai implementare esattamente questa fase: **quantizzare e dequantizzare** (dividere, arrotondare, moltiplicare di nuovo) — il cuore della compressione *lossy* del JPEG.

#### 📋 Linee Guida di Implementazione

1. **Dimensione del blocco:** Leggere l'intero $N$ (blocco $N \times N$).
2. **Coefficienti:** Leggere la matrice $C$ dei coefficienti DCT, $N$ righe con $N$ interi ciascuna (possono essere negativi).
3. **Tabella di quantizzazione:** Leggere la matrice $Q$, $N$ righe con $N$ interi positivi ciascuna.
4. **Quantizzazione:** Per ogni posizione $(u,v)$, calcolare l'indice quantizzato
$$
\tilde{C}(u,v) = \text{round}\!\left(\frac{C(u,v)}{Q(u,v)}\right)
$$
usando l'arrotondamento standard all'intero più vicino (i valori intermedi `.5` non si verificano mai nei casi di test).
5. **Dequantizzazione (ricostruzione):** Calcolare
$$
C'(u,v) = \tilde{C}(u,v) \times Q(u,v)
$$
6. **Output:** Visualizzare la matrice ricostruita $C'$, $N \times N$, di interi.

#### 📌 Vincoli Computazionali

* ***Round-trip* completo:** l'output è il coefficiente **ricostruito** ($\tilde{C} \times Q$), non l'indice quantizzato isolato.
* **Divisione in virgola mobile:** la divisione $C(u,v)/Q(u,v)$ deve essere eseguita in virgola mobile prima dell'arrotondamento — la divisione intera troncata produrrà un risultato errato.
* **Segno preservato:** i coefficienti negativi mantengono il segno dopo la quantizzazione e la ricostruzione.
* **$Q(u,v) > 0$ sempre:** non è necessario gestire la divisione per zero.

#### 🧠 Fondamenti Teorici

| Coefficiente | Frequenza | Valore tipico di $Q$ | Effetto della quantizzazione |
|---|---|---|---|
| $C(0,0)$ | DC (media del blocco) | Piccolo | Quasi sempre sopravvive — domina l'energia |
| $C(u,v)$ con $u+v$ basso | Bassa frequenza | Piccolo/medio | Parzialmente preservato |
| $C(u,v)$ con $u+v$ alto | Alta frequenza | Grande | Spesso diventa zero — fonte della compressione |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $N$.
* $N$ righe successive: matrice $C$ (coefficienti DCT, interi, possono essere negativi).
* $N$ righe successive: matrice $Q$ (tabella di quantizzazione, interi positivi).

**Output:**

* Matrice ricostruita $C'$, $N \times N$, interi separati da spazio.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 4<br>50 10 -5 0<br>8 -3 2 1<br>0 1 0 0<br>2 0 0 -1<br>2 5 7 8<br>4 7 8 11<br>6 8 11 12<br>9 11 12 14 | 50 10 -7 0<br>8 0 0 0<br>0 0 0 0<br>0 0 0 0 | $C(0,0)=50/2=25 \to 25\times2=50$ (preservato). $C(0,2)=-5/7\approx-0.71\to-1\to-1\times7=-7$. Invece $C(1,1)=-3/7\approx-0.43\to0$: azzerato dalla quantizzazione — la maggior parte del blocco diventa zero, illustrando la compattazione dell'energia nell'angolo superiore sinistro. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0503" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0503 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0503 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0503 button:hover { background: #e8dfcf; }
  #sim-ep0503 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0503_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0503_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0503_cell { width: 46px; height: 36px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP05_03: Quantizzazione DCT</span>
  <span class="sim-ep0503_pill">round(C / Q) &times; Q</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0503_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Scala di Q (Aggressività): <span id="sim-ep0503_vl_s" style="font-family:monospace; color:#26241d;">1.00&times;</span>
      </label>
    </div>
    
    <input id="sim-ep0503_sl_s" type="range" min="0.25" max="4" step="0.25" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Regola la scala di Q e osserva quanti coefficienti sopravvivono (diversi da zero) dopo il round-trip.
    </div>
  </div>

  <!-- Exibição das Grades 4x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0503_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Coefficienti DCT (C)
      </div>
      <div id="sim-ep0503_grid_c" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0503_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Ricostruito (round(C / Q) &middot; Q)
      </div>
      <div id="sim-ep0503_grid_r" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0503_debug" class="sim-ep0503_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep03(root){
    if (!root || root.dataset.sim05Ep03Init) return;
    root.dataset.sim05Ep03Init = "1";

    var C = [[50, 10, -5, 0], [8, -3, 2, 1], [0, 1, 0, 0], [2, 0, 0, -1]];
    var Qbase = [[2, 5, 7, 8], [4, 7, 8, 11], [6, 8, 11, 12], [9, 11, 12, 14]];

    var s   = root.querySelector('#sim-ep0503_sl_s');
    var sv  = root.querySelector('#sim-ep0503_vl_s');
    var gc  = root.querySelector('#sim-ep0503_grid_c');
    var gr  = root.querySelector('#sim-ep0503_grid_r');
    var dbg = root.querySelector('#sim-ep0503_debug');

    function cell(v, faded){
      var c = document.createElement('div');
      c.className = 'sim-ep0503_cell';
      if (faded) {
        c.style.cssText = 'background:#fafaf7; color:#8a8371; border:1px solid #e4dcc8;';
      } else {
        c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      }
      c.textContent = v;
      return c;
    }

    function render(){
      var scale = parseFloat(s.value);
      sv.innerHTML = scale.toFixed(2) + '&times;';
      gc.innerHTML = '';
      gr.innerHTML = '';
      var zeros = 0, total = 16;

      for (var i = 0; i < 4; i++){
        for (var j = 0; j < 4; j++){
          gc.appendChild(cell(C[i][j], false));
          var Q = Qbase[i][j] * scale;
          var q = Math.round(C[i][j] / Q);
          var rec = Math.round(q * Q);
          if (rec === 0) zeros++;
          gr.appendChild(cell(rec, rec === 0));
        }
      }

      dbg.textContent = 'Zeri: ' + zeros + ' / ' + total + '  |  Quanto maior a escala de Q, mais zeros — maior compressão, menor qualidade.';
    }

    s.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep03(){
    var root = document.getElementById('sim-ep0503');
    if (root) initSim05Ep03(root); else setTimeout(tryInitSim05Ep03, 200);
  }
  tryInitSim05Ep03();
})();
</script>
""")

**Figura 5.3:** Simulatore EP05_03: Quantizzazione DCT (*round-trip*)


<figure id="fig-05-sim-ep0503">
  <img src="imagens/fig-05-sim-ep0503.png" alt=" Simulatore EP05_03: Quantizzazione DCT (*round-trip*) " style="max-width:80%" />
  <figcaption><strong>Figura 5.3:</strong>  Simulatore EP05_03: Quantizzazione DCT (*round-trip*) </figcaption>
</figure>

In [ ]:
%%writefile EP05_03.py
# Codice Python

In [ ]:
TestSuite("EP05_03.py").run()

### EP05_04 🔴 Implementazione della DFT 2D a partire dalla definizione

Un laboratorio di ricerca in **astronomia computazionale** ha ricevuto, da una missione remota, un piccolo sensore sperimentale i cui dati grezzi non possono essere elaborati tramite librerie moderne di FFT — l'ambiente di validazione è isolato e consente solo operazioni aritmetiche di base. Il team deve **reimplementare la Trasformata di Fourier Discreta 2D a partire dalla definizione matematica stessa**, cella per cella, per poi confrontare bit per bit con `np.fft.fft2` in un altro ambiente.

Questo è l'esercizio più concettuale della lista: non ci sono scorciatoie. Dovrai implementare direttamente la doppia sommatoria della [Equação 5](#eq-05-dft), evidenziando *perché* la FFT esiste — e il costo computazionale che essa evita.

#### 📋 Linee guida per l'implementazione

1. **Dimensioni:** Leggere i numeri interi $M$ (righe) e $N$ (colonne) dell'immagine $f(x,y)$.
2. **Dati:** Leggere i valori interi di $f(x,y)$, riga per riga.
3. **DFT 2D:** Per ogni coppia di frequenze $(u,v)$ con $u=0,\ldots,M-1$ e $v=0,\ldots,N-1$, calcolare
$$
F(u,v) = \sum_{x=0}^{M-1}\sum_{y=0}^{N-1} f(x,y)\, e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$
usando l'identità di Eulero $e^{-j\theta} = \cos(\theta) - j\sin(\theta)$ per separare la parte reale e quella immaginaria — **non utilizzare alcuna funzione FFT predefinita**.
4. **Magnitudine:** Calcolare $|F(u,v)| = \sqrt{\text{Re}(F)^2 + \text{Im}(F)^2}$ e arrotondare all'intero più vicino.
5. **Output:** Visualizzare la matrice delle magnitudini arrotondate, $M \times N$, nello stesso ordine (senza `fftshift` — il componente DC rimane in $(0,0)$).

#### 📌 Vincoli computazionali

* **Vietato l'uso di librerie FFT:** l'implementazione deve calcolare esplicitamente le doppie sommatorie (cicli annidati), anche se più lenta.
* **Senza `fftshift`:** l'output mantiene la convenzione grezza della DFT, con il componente DC in $F(0,0)$ (angolo superiore sinistro).
* **Arrotondamento:** la magnitudine finale deve essere arrotondata all'intero più vicino; nei casi di test non vi è ambiguità `.5`.
* **Precisione:** piccoli errori di virgola mobile (ordine di $10^{-6}$) prima dell'arrotondamento sono previsti e non influenzano il risultato intero finale.

#### 🧠 Fondamenti teorici

| Elemento | Significato |
|---|---|
| $F(0,0)$ | Componente DC — somma di tutti i pixel, $F(0,0) = \sum f(x,y)$ |
| Parte reale $\text{Re}(F)$ | Proiezione del segnale sui coseni |
| Parte immaginaria $\text{Im}(F)$ | Proiezione del segnale sui seni |
| Complessità di questa implementazione | $\mathcal{O}((MN)^2)$ — ecco perché la FFT, con $\mathcal{O}(MN\log(MN))$, è indispensabile nelle immagini reali |

#### 📦 Specifica di input e output (VPL)

**Input:**

* Riga 1: intero $M$.
* Riga 2: intero $N$.
* Righe successive: elementi interi di $f(x,y)$, $M$ righe.

**Output:**

* Matrice delle magnitudini $|F(u,v)|$ arrotondate, $M \times N$, separate da spazio.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 2<br>2<br>1 2<br>3 4 | 10 2<br>4 0 | $F(0,0)=1+2+3+4=10$ (DC = somma totale). $F(0,1)=(1-2)+(3-4)=-2 \to |F|=2$. $F(1,0)=(1+2)-(3+4)=-4\to|F|=4$. $F(1,1)=(1-2)-(3-4)=0$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0504" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0504 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0504 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0504 button:hover { background: #e8dfcf; }
  .sim-ep0504_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0504_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0504_cell { width: 52px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 13px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP05_04: DFT 2D &mdash; Definizione Diretta</span>
  <span class="sim-ep0504_pill">&Sigma;&Sigma; f(x,y) e<sup>-j2&pi;(&hellip;)</sup></span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Instrução -->
  <div class="sim-ep0504_panel" style="margin-bottom:14px; text-align:center;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600;">
      Clicca sulle celle di f(x,y) per modificarne i valori (incrementa +1; Shift + clic decrementa -1) e osserva |F(u,v)| ricalcolato in tempo reale.
    </div>
  </div>

  <!-- Exibição das Grades 2x2 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0504_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        f(x,y) &mdash; Dominio Spaziale
      </div>
      <div id="sim-ep0504_grid_f" style="display:grid; grid-template-columns:repeat(2, 52px); gap:6px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0504_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        |F(u,v)| &mdash; Magnitudine (Senza Shift)
      </div>
      <div id="sim-ep0504_grid_F" style="display:grid; grid-template-columns:repeat(2, 52px); gap:6px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0504_debug" class="sim-ep0504_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep04(root){
    if (!root || root.dataset.sim05Ep04Init) return;
    root.dataset.sim05Ep04Init = "1";

    var f = [[1, 2], [3, 4]];
    var gf  = root.querySelector('#sim-ep0504_grid_f');
    var gF  = root.querySelector('#sim-ep0504_grid_F');
    var dbg = root.querySelector('#sim-ep0504_debug');

    function render(){
      gf.innerHTML = '';
      gF.innerHTML = '';

      for (var x = 0; x < 2; x++){
        for (var y = 0; y < 2; y++){
          (function(xx, yy){
            var c = document.createElement('div');
            c.className = 'sim-ep0504_cell';
            c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7; cursor:pointer;';
            c.textContent = f[xx][yy];
            c.addEventListener('click', function(e){
              if (e.shiftKey){ f[xx][yy]--; } else { f[xx][yy]++; }
              render();
            });
            gf.appendChild(c);
          })(x, y);
        }
      }

      var M = 2, N = 2;
      for (var u = 0; u < M; u++){
        for (var v = 0; v < N; v++){
          var re = 0, im = 0;
          for (var x = 0; x < M; x++){
            for (var y = 0; y < N; y++){
              var theta = 2 * Math.PI * (u * x / M + v * y / N);
              re += f[x][y] * Math.cos(theta);
              im -= f[x][y] * Math.sin(theta);
            }
          }
          var mag = Math.round(Math.sqrt(re * re + im * im));
          var c = document.createElement('div');
          c.className = 'sim-ep0504_cell';
          c.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          c.textContent = mag;
          gF.appendChild(c);
        }
      }

      dbg.textContent = 'F(0,0) = somma di tutti i pixel = ' + (f[0][0] + f[0][1] + f[1][0] + f[1][1]) + ' (componente DC)';
    }

    render();
  }

  function tryInitSim05Ep04(){
    var root = document.getElementById('sim-ep0504');
    if (root) initSim05Ep04(root); else setTimeout(tryInitSim05Ep04, 200);
  }
  tryInitSim05Ep04();
})();
</script>
""")

**Figura 5.4:** Simulatore EP05_04: DFT 2D manuale


<figure id="fig-05-sim-ep0504">
  <img src="imagens/fig-05-sim-ep0504.png" alt=" Simulatore EP05_04: DFT 2D manuale " style="max-width:80%" />
  <figcaption><strong>Figura 5.4:</strong>  Simulatore EP05_04: DFT 2D manuale </figcaption>
</figure>

In [ ]:
%%writefile EP05_04.py
# Codice Python

In [ ]:
TestSuite("EP05_04.py").run()

### EP05_05 🏆 *Pipeline* JPEG Completo: DCT, Quantizzazione e Ricostruzione

Sei stato incaricato di creare, da zero, un **codec JPEG didattico** in un ambiente embedded, senza alcuna libreria di immagini disponibile — solo operazioni matematiche di base. Il cliente vuole capire esattamente dove la qualità viene persa e dove viene recuperata, blocco per blocco. Questa è la sfida finale del capitolo: integrare **tutto** ciò che è stato studiato — la DCT-II ortonormale, la quantizzazione percettiva e la ricostruzione tramite IDCT — in un unico *pipeline* end-to-end, elaborando un blocco $N \times N$ dall'inizio alla fine, esattamente come fa internamente lo standard JPEG, $8\times8$ pixel alla volta.

#### 📋 Linee Guida di Implementazione

1. **Dimensione del blocco:** Leggere il numero intero $N$.
2. **Blocco originale:** Leggere la matrice di pixel $f(x,y)$, $N$ righe con $N$ interi in $[0,255]$.
3. **Tabella di quantizzazione:** Leggere la matrice $Q$, $N \times N$ interi positivi.
4. **Centratura:** Sottrarre 128 da ogni pixel: $g(x,y) = f(x,y) - 128$.
5. **DCT-II 2D ortonormale:** Calcolare
$$
C(u,v) = \alpha(u)\,\alpha(v)\sum_{x=0}^{N-1}\sum_{y=0}^{N-1} g(x,y)\,\cos\!\left[\frac{\pi(2x+1)u}{2N}\right]\cos\!\left[\frac{\pi(2y+1)v}{2N}\right]
$$
con $\alpha(0)=\sqrt{1/N}$ e $\alpha(k)=\sqrt{2/N}$ per $k>0$.
6. **Quantizzazione:** $\tilde{C}(u,v) = \text{round}(C(u,v)/Q(u,v))$.
7. **Dequantizzazione:** $C'(u,v) = \tilde{C}(u,v)\times Q(u,v)$.
8. **IDCT-II 2D (inversa ortonormale):** Calcolare $g'(x,y)$ da $C'(u,v)$ usando la trasformata inversa corrispondente (stessa base, sommatoria su $u,v$).
9. **Inversione della centratura e arrotondamento:** $f'(x,y) = \text{round}(g'(x,y) + 128)$, limitato all'intervallo $[0,255]$ (*clipping*).
10. **Output:** Visualizzare il blocco ricostruito $f'$, $N \times N$, interi.

#### 📌 Vincoli Computazionali

* ***Pipeline* completo obbligatorio:** tutte e sei le fasi (centrare, DCT, quantizzare, dequantizzare, IDCT, invertire) devono essere implementate — saltare la quantizzazione non supera i test, poiché il risultato sarebbe identico all'originale.
* ***Clipping*:** i valori ricostruiti al di fuori di $[0,255]$ devono essere troncati (0 se negativi, 255 se maggiori di 255).
* **Arrotondamento:** sia nella quantizzazione che nella ricostruzione finale dei pixel, usare l'arrotondamento standard; i casi di test evitano ambiguità `.5`.
* **Base ortonormale:** la normalizzazione $\alpha(u)$ e $\alpha(v)$ deve essere applicata esattamente come specificato — senza di essa, la IDCT non ricostruisce correttamente.

#### 🧠 Fondamenti Teorici

| Fase | Analoga nel vero standard JPEG | Dove la qualità viene persa |
|---|---|---|
| Centratura | Stessa — la DCT presuppone un segnale centrato su zero | Nessuna perdita |
| DCT-II | Fasi 3–4 del *pipeline* ([Tabela 5](#tbl-05-pipeline-jpeg)) | Nessuna perdita (trasformazione esatta e reversibile) |
| Quantizzazione | Fase 5 — divisione per $Q(u,v)$ | **Principale fonte di perdita** — i coefficienti ad alta frequenza diventano zero |
| IDCT | Ricostruzione finale | Ricostruisce esattamente i coefficienti *quantizzati*, non quelli originali |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $N$.
* $N$ righe successive: blocco originale $f(x,y)$, interi in $[0,255]$.
* $N$ righe successive: tabella di quantizzazione $Q$, interi positivi.

**Output:**

* Blocco ricostruito $f'(x,y)$, $N \times N$, interi in $[0,255]$, separati da spazi.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 4<br>120 130 125 128<br>115 140 135 122<br>118 150 160 130<br>110 120 145 138<br>4 6 8 10<br>6 8 10 12<br>8 10 12 16<br>10 12 16 20 | 118 126 119 131<br>114 143 140 119<br>117 149 159 130<br>107 121 146 139 | Dopo la DCT, una quantizzazione aggressiva sulle alte frequenze (valori grandi di $Q$ nell'angolo in basso a destra) e la ricostruzione tramite IDCT, il blocco risulta **vicino** all'originale, ma non identico — la differenza è il costo della compressione *lossy*. |

#### 💡 Suggerimento per il Debug

Se il risultato non corrisponde, verifica in questo ordine: (1) i coefficienti DCT grezzi (prima della quantizzazione) — devono ricostruire l'originale **esattamente** tramite IDCT se salti le fasi 6–7; (2) la tabella $\alpha(u)$ — un errore comune è applicare $\sqrt{2/N}$ anche per $u=0$; (3) l'arrotondamento della quantizzazione, che deve avvenire **prima** di moltiplicare di nuovo per $Q$.

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0505" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0505 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0505 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0505 button:hover { background: #e8dfcf; }
  #sim-ep0505 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0505_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0505_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0505_cell { width: 46px; height: 36px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP05_05: Pipeline JPEG (Blocco 4&times;4)</span>
  <span class="sim-ep0505_pill">DCT &rarr; Q &rarr; IDCT</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0505_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Scala di Q (1 = Tabella Base, Maggiore = Più Perdita): <span id="sim-ep0505_vl_s" style="font-family:monospace; color:#26241d;">1.00&times;</span>
      </label>
    </div>
    
    <input id="sim-ep0505_sl_s" type="range" min="0.5" max="5" step="0.5" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Regola il fattore di scala di quantizzazione e osserva il blocco ricostruito allontanarsi (o avvicinarsi) dall'originale.
    </div>
  </div>

  <!-- Exibição das Grades 4x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0505_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Blocco Originale
      </div>
      <div id="sim-ep0505_grid_o" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0505_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Ricostruito (DCT &rarr; Q &rarr; IDCT)
      </div>
      <div id="sim-ep0505_grid_r" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0505_debug" class="sim-ep0505_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep05(root){
    if (!root || root.dataset.sim05Ep05Init) return;
    root.dataset.sim05Ep05Init = "1";

    var N = 4;
    var f = [[120, 130, 125, 128], [115, 140, 135, 122], [118, 150, 160, 130], [110, 120, 145, 138]];
    var Qbase = [[4, 6, 8, 10], [6, 8, 10, 12], [8, 10, 12, 16], [10, 12, 16, 20]];

    var s   = root.querySelector('#sim-ep0505_sl_s');
    var sv  = root.querySelector('#sim-ep0505_vl_s');
    var go  = root.querySelector('#sim-ep0505_grid_o');
    var gr  = root.querySelector('#sim-ep0505_grid_r');
    var dbg = root.querySelector('#sim-ep0505_debug');

    function alpha(k){ return k === 0 ? Math.sqrt(1 / N) : Math.sqrt(2 / N); }

    function dct2(g){
      var C = [];
      for (var u = 0; u < N; u++){ C.push(new Array(N).fill(0)); }
      for (var u = 0; u < N; u++){
        for (var v = 0; v < N; v++){
          var sum = 0;
          for (var x = 0; x < N; x++){
            for (var y = 0; y < N; y++){
              sum += g[x][y] * Math.cos(Math.PI * (2 * x + 1) * u / (2 * N)) * Math.cos(Math.PI * (2 * y + 1) * v / (2 * N));
            }
          }
          C[u][v] = alpha(u) * alpha(v) * sum;
        }
      }
      return C;
    }

    function idct2(C){
      var g = [];
      for (var x = 0; x < N; x++){ g.push(new Array(N).fill(0)); }
      for (var x = 0; x < N; x++){
        for (var y = 0; y < N; y++){
          var sum = 0;
          for (var u = 0; u < N; u++){
            for (var v = 0; v < N; v++){
              sum += alpha(u) * alpha(v) * C[u][v] * Math.cos(Math.PI * (2 * x + 1) * u / (2 * N)) * Math.cos(Math.PI * (2 * y + 1) * v / (2 * N));
            }
          }
          g[x][y] = sum;
        }
      }
      return g;
    }

    function cell(v){
      var c = document.createElement('div');
      c.className = 'sim-ep0505_cell';
      c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      c.textContent = v;
      return c;
    }

    function render(){
      var scale = parseFloat(s.value);
      sv.innerHTML = scale.toFixed(2) + '&times;';
      go.innerHTML = '';
      gr.innerHTML = '';

      var g = [];
      for (var x = 0; x < N; x++){
        var row = [];
        for (var y = 0; y < N; y++){
          row.push(f[x][y] - 128);
        }
        g.push(row);
      }

      var C = dct2(g);
      var Cq = [];
      for (var u = 0; u < N; u++){
        var row = [];
        for (var v = 0; v < N; v++){
          var Qv = Qbase[u][v] * scale;
          var q = Math.round(C[u][v] / Qv);
          row.push(q * Qv);
        }
        Cq.push(row);
      }

      var gr2 = idct2(Cq);
      var diffSum = 0, n = 0;

      for (var x = 0; x < N; x++){
        for (var y = 0; y < N; y++){
          go.appendChild(cell(f[x][y]));
          var rec = Math.round(gr2[x][y] + 128);
          rec = Math.max(0, Math.min(255, rec));
          gr.appendChild(cell(rec));
          diffSum += Math.abs(rec - f[x][y]);
          n++;
        }
      }

      dbg.textContent = 'Errore medio assoluto per pixel: ' + (diffSum / n).toFixed(2) + '  |  Quanto maior a escala de Q, maior o erro de reconstrução.';
    }

    s.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep05(){
    var root = document.getElementById('sim-ep0505');
    if (root) initSim05Ep05(root); else setTimeout(tryInitSim05Ep05, 200);
  }
  tryInitSim05Ep05();
})();
</script>
""")

**Figura 5.5:** Simulatore EP05_05: *Pipeline* JPEG completo a blocchi


<figure id="fig-05-sim-ep0505">
  <img src="imagens/fig-05-sim-ep0505.png" alt=" Simulatore EP05_05: *Pipeline* JPEG completo a blocchi " style="max-width:80%" />
  <figcaption><strong>Figura 5.5:</strong>  Simulatore EP05_05: *Pipeline* JPEG completo a blocchi </figcaption>
</figure>

In [ ]:
%%writefile EP05_05.py
# Codice Python

In [ ]:
TestSuite("EP05_05.py").run()